<a href="https://colab.research.google.com/github/pranay8297/Stable-Diffusion-Experiments/blob/main/img_quality_assesment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Try - Claude

In [ ]:
!pip install transformers torch accelerate bitsandbytes

In [2]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id)

In [4]:
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [20]:
from PIL import Image
prompt_text = '''This is an image of a utility meter with a digital display. Evaluate whether this image is suitable for accurate Optical Character Recognition (OCR) processing.

Please answer 'yes' or 'no' AND explain your reasoning based on these criteria:

For a 'yes' answer, ALL these conditions must be met:
- The complete meter display area is visible in the frame
- All digits are clearly visible with sharp definition
- No significant blur affecting character recognition
- No glare or reflections obscuring any digits
- Sufficient resolution where individual digits are crisp
- Proper orientation of the display
- Strong contrast between digits and background

For a 'no' answer, explain which specific issues are present:
- Parts of the display cut off or out of frame
- Blurry, pixelated, or low resolution image
- Glare reflecting off the display
- Partially obscured or unclear digits
- Poor lighting making digits difficult to distinguish
- Display at an angle that distorts digit appearance
- Poor contrast making digits blend with background

First state your answer ('yes' or 'no'), then provide a brief explanation of the specific factors that led to your decision. Be very strict in your evaluation - when in doubt, answer 'no'.'''


raw_image = Image.open("/content/not_possible.jpg")

conversation = [
    {

      "role": "user",
      "content": [
          {"type": "text", "text": prompt_text},
          {"type": "image"},
        ],
    },
]
prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
inputs = processor(images=raw_image, text=prompt, return_tensors='pt').to(0, torch.float16)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=200, do_sample=False)

print(processor.decode(output[0], skip_special_tokens=True))

USER:  
This is an image of a utility meter with a digital display. Evaluate whether this image is suitable for accurate Optical Character Recognition (OCR) processing.

Please answer 'yes' or 'no' AND explain your reasoning based on these criteria:

For a 'yes' answer, ALL these conditions must be met:
- The complete meter display area is visible in the frame
- All digits are clearly visible with sharp definition
- No significant blur affecting character recognition
- No glare or reflections obscuring any digits
- Sufficient resolution where individual digits are crisp
- Proper orientation of the display
- Strong contrast between digits and background

For a 'no' answer, explain which specific issues are present:
- Parts of the display cut off or out of frame
- Blurry, pixelated, or low resolution image
- Glare reflecting off the display
- Partially obscured or unclear digits
- Poor lighting making digits difficult to distinguish
- Display at an angle that distorts digit appearance
- 

In [55]:
from PIL import Image

# Simplify the prompt structure
prompt_text = """This is an image of a utility meter. Your task is to evaluate if this image is suitable for Optical Character Recognition (OCR) processing - specifically whether the meter reading can be accurately extracted by an OCR system. \
    \
    Answer ONLY with 'yes' or 'no' based on the following strict criteria: \
    \
    Answer 'yes' ONLY if ALL of these conditions are met: \
    - The ENTIRE meter display area is visible in the frame \
    - ALL digits in the display are clearly visible with sharp definition \
    - There is NO significant blur that would impede character recognition \
    - There is NO glare or reflection obscuring any part of the digits \
    - The image has sufficient resolution where individual digits are crisp \
    - The display is properly oriented (not tilted or upside down) \
    - The contrast between digits and background is strong \
    \
    Answer 'no' if ANY of these conditions are present: \
    - Parts of the meter display are cut off or out of frame \
    - The image is blurry, pixelated, or low resolution \
    - There is glare reflecting off the display \
    - Some digits are partially obscured or unclear \
    - The image is poorly lit making digits difficult to distinguish \
    - The display is at an angle that distorts digit appearance \
    - The contrast is poor making digits blend with background \
    \
    Examples: \
    - An image showing the complete meter with all digits clearly visible, good lighting, no blur or glare = 'yes' \
    - An image with slight blur but all digits still distinctly recognizable = 'yes' \
    - An image with significant blur where digits appear smudged = 'no' \
    - An image with glare covering part of the display = 'no' \
    - An image where part of the meter display is cut off = 'no' \
    - An image where the digits are clear but very small/low resolution = 'no' \
    \
    Remember to be VERY STRICT in your evaluation. When in doubt, answer 'no'."""


# For LLAVA specifically, try using a simpler input format
raw_image = Image.open(list(path.iterdir())[5])

conversation = [
    {

      "role": "user",
      "content": [
          {"type": "text", "text": prompt_text},
          {"type": "image"},
        ],
    },
]
prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
inputs = processor(images=raw_image, text=prompt, return_tensors='pt').to(0, torch.float16)

# Generate with more tokens to get the full explanation
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=512,  # Increase this to get full explanation
        do_sample=False,
        temperature=0.0,     # Make response deterministic
    )

# Decode properly - don't slice the output tokens
response = processor.decode(output[0], skip_special_tokens=True)

# Remove the prompt from the response if needed
if prompt_text in response:
    response = response.replace(prompt_text, "").strip()

print(response)

USER:  
 ASSISTANT: Yes


In [ ]:
!unzip /content/20250302.zip

In [56]:
def get_label(image_path):
    prompt_text = "This is an image of a utility meter with a digital display. Evaluate whether this image is suitable for accurate Optical Character Recognition (OCR) processing. Answer with 'yes' or 'no' and explain why based on image quality, visibility of digits, blur, glare, resolution, and whether the complete display is visible. Be very strict - when in doubt, answer 'no'."


    # For LLAVA specifically, try using a simpler input format
    raw_image = Image.open(image_path)

    conversation = [
        {

        "role": "user",
        "content": [
            {"type": "text", "text": prompt_text},
            {"type": "image"},
            ],
        },
    ]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=raw_image, text=prompt, return_tensors='pt').to(0, torch.float16)

    # Generate with more tokens to get the full explanation
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,  # Increase this to get full explanation
            do_sample=False,
            temperature=0.0,     # Make response deterministic
        )

    # Decode properly - don't slice the output tokens
    response = processor.decode(output[0], skip_special_tokens=True)

    # Remove the prompt from the response if needed
    if prompt_text in response:
        response = response.replace(prompt_text, "").strip()

    label = response.rsplit(':')[-1].strip()
    print(label)
    return label

In [57]:
from pathlib import Path
from tqdm import tqdm

In [ ]:
path = Path('/content/20250302')
labels = {}
for img_path in tqdm(list(path.iterdir())):
    label = get_label(img_path)
    labels[img_path.name] = label

In [66]:
import pandas as pd
paths = list(labels.keys())
labels = list(labels.values())
df = pd.DataFrame({'path': paths, 'label': labels})

In [67]:
df.head()

,path,label
0,2307010904_20250302183223_895_Premisephoto1.jpg,No
1,2775010006_20250302104946_544_NewMeterInitialr...,Yes
2,2444003732_20250302132738_944_MeteringPhoto.jpg,No
3,2345000490_20250302133905_488_Premisephoto1.jpg,Yes
4,2023023537_20250302110121_631_NewMeterInitialr...,Yes


In [68]:
df.to_csv('img_quality.csv')